# Phase B — Pipeline (leakage-safe data preparation)

CRISP-DM *Data Preparation*. Produces the **model-ready, leakage-safe** datasets that **all teammates
train on**, so the 6 models are comparable in Phase C.

**Outputs**
Features = 3 categorical columns (`genre`, `platform_family`, `publisher_tier`). **Not** encoded/scaled,
each teammate applies their own encoding / scaling / balancing / tuning inside their own pipeline.

**Leakage-safe principle:** split first, compute `Publisher_Tier` from **train only**, test gets a
lookup (unseen publisher → tier 0). The `Hit` label is the *target definition* (year-based percentile),
so it is computed on the full scope data before the stratified split.

In [1]:
import os, warnings; warnings.filterwarnings('ignore')
import pandas as pd, numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
from IPython.display import display

df = pd.read_csv("data/raw/Video Games Sales (1980-2024) - Raw.csv")
df['year'] = pd.to_datetime(df['release_date'], format='%d-%m-%Y', errors='coerce').dt.year

# Locked cleaning (same as Phase A)
FAM = {}
for x in ['PS','PS2','PS3','PS4','PS5','PSP','PSV','PSN']: FAM[x]='PlayStation'
for x in ['XB','X360','XOne','XS','XBL']: FAM[x]='Xbox'
for x in ['NS','Wii','WiiU','DS','DSi','DSiW','3DS','GB','GBC','GBA','SNES','NES','N64','GC','VB','VC','FDS','iQue']: FAM[x]='Nintendo'
for x in ['PC','OSX','Linux']: FAM[x]='PC'
df['platform_family'] = df['console'].map(FAM).fillna('Other')
df['genre'] = df['genre'].replace(['Education','Board Game','Sandbox','Party','MMO'], 'Misc')
print(f"Loaded {len(df):,} rows.")

Loaded 64,016 rows.


## The leakage-safe `prepare()` routine

In [2]:
FEATS = ['genre', 'platform_family', 'publisher_tier']

def publisher_tier_lookup(train):
    """Train-only: tier = # of a publisher's million-seller (>=1.0M) games (de-duped by title)."""
    u = train.drop_duplicates(['title', 'publisher']).assign(ic=lambda x: (x['total_sales'] >= 1.0).astype(int))
    cnt = u.groupby('publisher')['ic'].sum()
    tier = lambda n: 0 if n <= 0 else (1 if n <= 4 else 2)   # 0 / 1-4 / 5+
    return {pub: tier(n) for pub, n in cnt.items()}

def prepare(name, year_lo, year_hi=2018, q_hit=0.20):
    sub = df[(df['year'].between(year_lo, year_hi)) & df['total_sales'].notna()].copy()
    # Target: Hit = top q% within release year (target definition -> on full scope data)
    thr = sub.groupby('year')['total_sales'].transform(lambda s: s.quantile(1 - q_hit))
    sub['Hit'] = (sub['total_sales'] >= thr).astype(int)
    # Stratified split FIRST
    train, test = train_test_split(sub, test_size=0.2, random_state=42, stratify=sub['Hit'])
    # Publisher_Tier from TRAIN ONLY; test = lookup, unseen publisher -> tier 0
    pt = publisher_tier_lookup(train)
    train['publisher_tier'] = train['publisher'].map(pt).fillna(0).astype(int)
    test['publisher_tier']  = test['publisher'].map(pt).fillna(0).astype(int)
    # Save
    out = f"data/processed/{name}"; os.makedirs(out, exist_ok=True)
    train[FEATS].to_csv(f"{out}/X_train.csv", index=False)
    test[FEATS].to_csv(f"{out}/X_test.csv", index=False)
    train[['Hit']].to_csv(f"{out}/y_train.csv", index=False)
    test[['Hit']].to_csv(f"{out}/y_test.csv", index=False)
    # meta = NOT features; for Phase C only (threshold 20-vs-30 & scope analysis)
    train[['title','year','total_sales']].to_csv(f"{out}/meta_train.csv", index=False)
    test[['title','year','total_sales']].to_csv(f"{out}/meta_test.csv", index=False)
    return train, test

## Build the two scopes

`scope_2000_2018` = primary (teammates train on this). `scope_all` = no lower cutoff (≤2018, incl. pre-2000),
for the Phase-C scope comparison, scored on identical test rows there.

In [3]:
built = {}
built['scope_2000_2018'] = prepare('scope_2000_2018', 2000, 2018)
built['scope_all']       = prepare('scope_all',       1980, 2018)

summary = []
for name, (tr, te) in built.items():
    summary.append({'scope': name, 'train': len(tr), 'test': len(te),
                    'train_Hit%': round(tr['Hit'].mean()*100, 1),
                    'test_Hit%': round(te['Hit'].mean()*100, 1),
                    'features': len(FEATS)})
display(pd.DataFrame(summary))
for name, (tr, te) in built.items():
    print(f"{name:18} train publisher_tier dist: {tr['publisher_tier'].value_counts().sort_index().to_dict()}")

,scope,train,test,train_Hit%,test_Hit%,features
0,scope_2000_2018,13681,3421,20.3,20.3,3
1,scope_all,15006,3752,20.4,20.4,3


scope_2000_2018    train publisher_tier dist: {0: 4381, 1: 2410, 2: 6890}
scope_all          train publisher_tier dist: {0: 4375, 1: 2581, 2: 8050}


## Temporal-leakage check (honesty test)

The split is random, so a publisher's tier could be set by its *future* games. We test it: train on
≤2014, predict 2015–2018, with tier built from the **past only**. If the temporal AUC is close to the
random-split AUC, the random-split numbers are not inflated by future leakage.

In [4]:
def quick_auc(train, test):
    X = pd.get_dummies(pd.concat([train[FEATS], test[FEATS]]).astype(str))
    rf = RandomForestClassifier(n_estimators=300, max_depth=7, random_state=42, class_weight='balanced')
    rf.fit(X.iloc[:len(train)], train['Hit'])
    return roc_auc_score(test['Hit'], rf.predict_proba(X.iloc[len(train):])[:, 1])

# (a) random split — our processed scope_2000_2018
tr_r, te_r = built['scope_2000_2018']
auc_random = quick_auc(tr_r, te_r)

# (b) temporal split — train<=2014, test 2015-18, tier from past only
sub = df[(df['year'].between(2000, 2018)) & df['total_sales'].notna()].copy()
thr = sub.groupby('year')['total_sales'].transform(lambda s: s.quantile(0.80)); sub['Hit'] = (sub['total_sales'] >= thr).astype(int)
tr_t = sub[sub['year'] <= 2014].copy(); te_t = sub[sub['year'] >= 2015].copy()
pt = publisher_tier_lookup(tr_t)
tr_t['publisher_tier'] = tr_t['publisher'].map(pt).fillna(0).astype(int)
te_t['publisher_tier'] = te_t['publisher'].map(pt).fillna(0).astype(int)
auc_temporal = quick_auc(tr_t, te_t)

print(f"Random-split   test AUC : {auc_random:.3f}")
print(f"Temporal-split test AUC : {auc_temporal:.3f}   (train<=2014, test 2015-18; tier from past only)")
print(f"Gap: {auc_random - auc_temporal:+.3f}  ->  "
      + ("no major temporal inflation; random-split numbers are honest." if abs(auc_random-auc_temporal) < 0.05
         else "some gap -> partly newer-era shift / partly reputation built from future (note in report)."))

Random-split   test AUC : 0.781
Temporal-split test AUC : 0.829   (train<=2014, test 2015-18; tier from past only)
Gap: -0.048  ->  no major temporal inflation; random-split numbers are honest.
